# Capítulo 3 · Memoria del repositorio e inicialización

Las tareas se cortan en sesiones de doce pasos, y cada sesión empieza con la conversación vacía. Se comparan cuatro formas de trabajar: una sola sesión de 24 pasos, dos sesiones sin estado, dos sesiones con un archivo de progreso que escribe el agente y dos sesiones con un informe de traspaso que prepara el programa.

**Antes de empezar, activa la GPU:** menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

## 1. Cargar el modelo, el repositorio y las herramientas

In [ ]:
!pip install -q transformers==5.19.0  # las versiones con las que se comprobó el libro

In [ ]:
"""Capítulo 3: memoria del repositorio e inicialización."""
import importlib
import io
import json
import re
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_GPU = "Qwen/Qwen3-4B-Instruct-2507"  # con la GPU T4 gratuita de Colab
MODELO_CPU = "Qwen/Qwen3-1.7B"              # sin GPU: más pequeño, y peor agente
RAIZ = Path.cwd()                           # donde están datos/ y comun_libro/
REPOSITORIO = ("https://github.com/HernanDiaz/ai-agents-real-projects/"
               "archive/refs/heads/main.zip")
TAREAS = ["T01", "T02", "T03", "T04", "T05", "T10", "T11", "T13", "T14", "T15",
          "T17", "T18", "T19", "T20"]       # las tareas de una sola sesión
MAX_PASOS = 30             # llamadas al modelo por tarea
MAX_TOKENS_PASO = 600      # tokens nuevos por llamada
LIMITE_ENTRADA = 12_000    # tokens de la conversación: más no cabe en la T4
RESULTADOS_INTACTOS = 2    # los últimos resultados de herramientas no se recortan
MAX_RECHAZOS = 3           # veces que la puerta devuelve el trabajo
hr = None                  # el módulo de herramientas, tras preparar()



def cargar(nombre=None):
    """Carga el modelo y su tokenizador: en la GPU, en float16; si no, en la CPU."""
    gpu = torch.cuda.is_available()
    if nombre is None:
        nombre = MODELO_GPU if gpu else MODELO_CPU
    tok = AutoTokenizer.from_pretrained(nombre)
    modelo = AutoModelForCausalLM.from_pretrained(
        nombre, dtype=torch.float16 if gpu else torch.float32)
    modelo.to("cuda" if gpu else "cpu").eval()
    print(f"Modelo {nombre} en {'cuda' if gpu else 'cpu'}")
    return tok, modelo


def preparar():
    """El repositorio de Ciclos Norte, sus tareas y las herramientas del agente.
    Si no están (en Colab), se descargan del repositorio del libro."""
    global hr
    if not (RAIZ / "comun_libro" / "herramientas_repo.py").exists():
        with urllib.request.urlopen(REPOSITORIO) as respuesta:
            archivo = zipfile.ZipFile(io.BytesIO(respuesta.read()))
        for nombre in archivo.namelist():
            relativa = nombre.split("/", 1)[1]
            if relativa.startswith(("datos/", "comun_libro/")) and not nombre.endswith("/"):
                destino = RAIZ / relativa
                destino.parent.mkdir(parents=True, exist_ok=True)
                destino.write_bytes(archivo.read(nombre))
    sys.path.insert(0, str(RAIZ / "comun_libro"))
    hr = importlib.import_module("herramientas_repo")
    print(f"Repositorio en {hr.REPO}: {len(hr.archivos_de(hr.REPO))} archivos")

In [ ]:
tok, modelo = cargar()
preparar()

## 2. El modelo

In [ ]:
def plantilla(tok, mensajes):
    """La conversación como texto, con las herramientas, en la plantilla del modelo."""
    return tok.apply_chat_template(mensajes, tools=hr.ESQUEMAS, tokenize=False,
                                   add_generation_prompt=True)


def generar(tok, modelo, texto):
    """Una respuesta del modelo, con decodificación voraz. Devuelve el texto y los
    tokens de entrada y de salida."""
    entrada = tok(texto, return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=MAX_TOKENS_PASO,
                                 do_sample=False, temperature=None, top_p=None,
                                 top_k=None, pad_token_id=tok.eos_token_id)
    n_entrada = entrada["input_ids"].shape[1]
    nuevos = salida[0, n_entrada:].cpu()
    del entrada, salida
    if torch.cuda.is_available():
        torch.cuda.empty_cache()  # cada paso tiene otra longitud: sin esto la
        #                           memoria reservada crece hasta llenar la GPU
    return tok.decode(nuevos, skip_special_tokens=True).strip(), n_entrada, len(nuevos)


def leer_llamadas(respuesta):
    """Las llamadas {name, arguments} que el modelo escribe entre <tool_call> y
    </tool_call>. Una llamada que no se entiende queda con name None."""
    llamadas = []
    for bloque in re.findall(r"<tool_call>(.*?)(?:</tool_call>|$)", respuesta, re.S):
        try:
            datos = json.loads(bloque.strip(), strict=False)
        except ValueError:
            datos = None
        if isinstance(datos, dict) and "name" in datos:
            argumentos = datos.get("arguments", {})
            llamadas.append({"name": datos["name"],
                             "arguments": argumentos if isinstance(argumentos, dict)
                             else {}})
        else:
            llamadas.append({"name": None, "arguments": {}})
    return llamadas


def recortar_conversacion(tok, mensajes):
    """Si la conversación pasa de LIMITE_ENTRADA tokens, cambia los resultados de
    herramientas más antiguos por un aviso corto. Devuelve el texto para el modelo y
    cuántos ha recortado (None si ni así cabe)."""
    texto = plantilla(tok, mensajes)
    recortados = 0
    candidatos = [m for m in mensajes if m["role"] == "tool"
                  and not m.get("recortado")][:-RESULTADOS_INTACTOS or None]
    while len(tok(texto)["input_ids"]) > LIMITE_ENTRADA:
        if not candidatos:
            return texto, None
        mensaje = candidatos.pop(0)
        mensaje["content"] = json.dumps(
            {"aviso": "Resultado antiguo recortado para ahorrar memoria; vuelve a "
                      "pedirlo si lo necesitas.",
             "inicio": mensaje["content"][:150]}, ensure_ascii=False)
        mensaje["recortado"] = True
        recortados += 1
        texto = plantilla(tok, mensajes)
    return texto, recortados

## 3. Las reglas del capítulo 2

In [ ]:
SISTEMA = (
    "Eres un agente de programación que trabaja en el repositorio de Ciclos Norte, "
    "una tienda de bicicletas de Oviedo. Es un programa de gestión en Python con "
    "varios módulos (pedidos, devoluciones, existencias, facturas, avisos…), sus "
    "pruebas en la carpeta tests/ y unos datos de ejemplo. Recibirás una petición de "
    "alguien de la tienda y tienes que resolverla cambiando el código con las "
    "herramientas: listar archivos, leer por rangos de líneas, buscar, editar por "
    "sustitución exacta, ejecutar las pruebas y unas pocas órdenes permitidas. "
    "Trabaja paso a paso, con una herramienta cada vez. Cuando hayas terminado, "
    "responde sin llamar a ninguna herramienta, empezando por «TERMINADO:» y "
    "explicando en una o dos frases qué has cambiado. Si no puedes terminar, empieza "
    "por «NO TERMINADO:» y explica por qué.")


AGENTS_REGLAS = """# AGENTS.md · Ciclos Norte

Reglas del repositorio. Cada una se comprueba al final de tu trabajo.

R1. Antes de editar un archivo, léelo con leer_archivo.
R2. No cambies nada de datos/.
R3. Cambia docs/ solo si la petición pide cambiar las condiciones generales.
R4. No borres ni cambies pruebas que ya existen. Excepción: si la petición cambia una
    regla del negocio, actualiza las pruebas de la regla antigua y dilo en tu respuesta.
R5. Añade tus pruebas en el archivo del módulo (tests/test_envios.py para envios.py):
    sustituye la línea `if __name__ == "__main__":` por una clase nueva con tu prueba,
    seguida de esa misma línea. Importa dentro de la prueba lo que necesites.
R6. No crees módulos nuevos fuera de tests/.
R7. «TERMINADO» exige: cambios en el código del programa, una comprobación nueva
    en tests/ y ejecutar_pruebas en verde después de tu último cambio.
R8. Tu respuesta final nombra todos los archivos que has cambiado.
"""


PIDE_DOCS = re.compile(r"(actualiz\w*|y en las)\s[^.]{0,40}condiciones\s+generales", re.I)


def lineas_del_diff(diff):
    """Las líneas añadidas y quitadas de cada archivo de un diff unificado:
    {ruta: {"+": [...], "-": [...]}}."""
    archivos, actual = {}, None
    for linea in diff.splitlines():
        if linea.startswith("+++ b/"):
            actual = archivos.setdefault(linea[6:], {"+": [], "-": []})
        elif linea.startswith("--- a/") or actual is None:
            continue
        elif linea[:1] in "+-" and linea[1:].strip():
            actual[linea[0]].append(linea[1:])
    return archivos


def comprobar_reglas(tarea, registro, cambios, respuesta):
    """Las reglas incumplidas, como {regla: evidencia}. Mira lo que de verdad ha
    cambiado (el diff), el registro de llamadas y la respuesta final."""
    rotas = {}
    lineas = lineas_del_diff(cambios["diff"])
    leidos = set()
    for r in registro:
        ruta = r["argumentos"].get("ruta", "")
        if r["herramienta"] == "leer_archivo" and r["ok"]:
            leidos.add(ruta)
        elif r["herramienta"] == "editar" and r["ok"] and ruta not in leidos:
            rotas.setdefault("R1", f"editaste {ruta} sin leerlo antes")
    for ruta in cambios["cambiados"]:
        if ruta.startswith("datos/"):
            rotas["R2"] = f"cambiaste {ruta}"
        if ruta.startswith("docs/") and not PIDE_DOCS.search(tarea["peticion"]):
            rotas["R3"] = f"cambiaste {ruta} y la petición no lo pide"
        if ruta.startswith("tests/") and ruta in cambios["nuevos"]:
            rotas["R5"] = f"creaste {ruta}; las pruebas van en el archivo del módulo"
        if not ruta.startswith(("tests/", "docs/", "datos/")) and ruta in cambios["nuevos"]:
            rotas["R6"] = f"creaste el módulo {ruta}"
    quitadas = [r for r, l in lineas.items() if r.startswith("tests/") and l["-"]]
    if quitadas and not re.search(r"prueba", respuesta or "", re.I):
        rotas["R4"] = (f"cambiaste pruebas que ya existían en {', '.join(quitadas)} "
                       "y tu respuesta no lo dice")
    if respuesta and dice_terminado(respuesta):
        programa = [r for r in cambios["cambiados"]
                    if not r.startswith(("tests/", "docs/", "datos/"))]
        prueba_nueva = any(re.search(r"def test_|assert", l)
                           for r, l2 in lineas.items() if r.startswith("tests/")
                           for l in l2["+"])
        ediciones = [i for i, r in enumerate(registro)
                     if r["herramienta"] in ("editar", "crear_archivo") and r["ok"]]
        despues = registro[ediciones[-1] + 1:] if ediciones else registro
        pruebas = [r for r in despues if r["herramienta"] == "ejecutar_pruebas"]
        falta = [texto for cumple, texto in (
            (programa, "no hay cambios en el código del programa"),
            (prueba_nueva, "no hay ninguna comprobación nueva en tests/"),
            (pruebas and pruebas[-1]["resultado"].get("todas_pasan"),
             "no hay una ejecución de las pruebas en verde después del último cambio"))
            if not cumple]
        if falta:
            rotas["R7"] = "; ".join(falta)
        sin_nombrar = [r for r in cambios["cambiados"]
                       if Path(r).name not in respuesta]
        if sin_nombrar:
            rotas["R8"] = f"tu respuesta no nombra {', '.join(sin_nombrar)}"
    return rotas


def dice_terminado(respuesta):
    """True si la respuesta final dice que la tarea está hecha. Una respuesta vacía
    no dice nada."""
    if not respuesta.strip():
        return False
    if re.search(r"NO\s+TERMINADO", respuesta, re.I):
        return False
    if re.search(r"TERMINADO", respuesta):
        return True
    return not re.search(r"no (he podido|puedo|consigo|es posible)|necesito|\?",
                         respuesta, re.I)


class HerramientasConReglas:
    """Las herramientas del agente con las reglas de proceso comprobadas antes de
    cada edición: si la edición incumple R1, R2, R3, R5 o R6, no se hace y el agente
    recibe la regla. El resto de llamadas pasan tal cual."""

    def __init__(self, herramientas, tarea):
        self.herramientas = herramientas
        self.registro = herramientas.registro
        self.tarea = tarea
        self.leidos = set()

    def regla_rota(self, nombre, ruta):
        """La regla que incumpliría editar o crear «ruta», o None."""
        existe = self.herramientas.espacio.ruta(ruta).exists()
        if ruta.startswith("datos/"):
            return "R2: no cambies nada de datos/."
        if ruta.startswith("docs/") and not PIDE_DOCS.search(self.tarea["peticion"]):
            return "R3: la petición no pide cambiar las condiciones generales."
        if nombre == "crear_archivo" and ruta.startswith("tests/"):
            return ("R5: no crees archivos de pruebas; añade tu prueba al archivo del "
                    "módulo, sustituyendo la línea if __name__ == \"__main__\":")
        if nombre == "crear_archivo" and not existe and not ruta.startswith(".agente/"):
            return "R6: no crees módulos nuevos fuera de tests/."
        if nombre == "editar" and existe and ruta not in self.leidos:
            return f"R1: lee {ruta} con leer_archivo antes de editarlo."
        return None

    def llamar(self, nombre, argumentos):
        """Como Herramientas.llamar, pero comprobando las reglas antes de editar."""
        ruta = str((argumentos or {}).get("ruta", ""))
        if nombre in ("editar", "crear_archivo"):
            regla = self.regla_rota(nombre, ruta)
            if regla:
                resultado = {"error": "Edición rechazada por AGENTS.md. " + regla}
                self.registro.append({"herramienta": nombre, "argumentos": argumentos,
                                      "ok": False, "segundos": 0,
                                      "resultado": resultado})
                return resultado
        resultado = self.herramientas.llamar(nombre, argumentos)
        if nombre == "leer_archivo" and "error" not in resultado:
            self.leidos.add(ruta)
        return resultado

## 4. El progreso y el traspaso

El informe de traspaso se puede probar sin modelo, sobre una copia con la solución de una tarea.

In [ ]:
PROGRESO = ".agente/PROGRESO.md"


REGLA_PROGRESO = """R9. Mantén .agente/PROGRESO.md con tres apartados: «Hecho», «Pendiente» y
    «Cómo comprobarlo». Créalo al empezar y actualízalo después de cada cambio: la
    sesión puede cortarse en cualquier momento y otra sesión seguirá desde ahí.
"""


AVISO_FIN = (f"Quedan dos pasos en esta sesión. Actualiza {PROGRESO} con lo hecho, lo "
             "pendiente y cómo comprobarlo, para que la sesión siguiente pueda seguir.")


def traspaso(espacio, ultimo_mensaje):
    """El informe de traspaso con el que empieza la sesión siguiente: qué ha cambiado
    en el repositorio (el diff), cómo están las pruebas y lo último que escribió el
    agente en la sesión anterior. Lo prepara el programa, sin el modelo y sin depender
    de que el agente haya tomado notas."""
    cambios = hr.diferencias(espacio.origen, espacio.carpeta)
    pruebas = hr.Herramientas(espacio).ejecutar_pruebas()
    lineas = ["Informe de traspaso (lo prepara el programa al empezar la sesión):",
              "- Archivos cambiados respecto al original: "
              + (", ".join(cambios["cambiados"]) or "ninguno") + "."]
    if cambios["diff"]:
        lineas.append("- Cambios hechos hasta ahora (diff):\n" + cambios["diff"][:2000])
    lineas.append(f"- Pruebas: {pruebas['resultado']} ({pruebas['ejecutadas']} ejecutadas).")
    if not pruebas["todas_pasan"]:
        lineas.append("  Final de la salida de las pruebas:\n" + pruebas["salida"][-800:])
    if ultimo_mensaje:
        lineas.append("- Lo último que escribió la sesión anterior:\n" + ultimo_mensaje[:600])
    return "\n".join(lineas)

In [ ]:
tarea = hr.leer_tarea("T01")
with hr.Espacio() as espacio:
    hr.aplicar_solucion(hr.Herramientas(espacio), tarea["carpeta"] / "solucion")
    print(traspaso(espacio, "He cambiado gastos_envio; falta comprobarlo."))

## 5. Varias sesiones

La tarea del nuevo precio de la revisión, sin estado y con traspaso. Compara las primeras llamadas de la segunda sesión.

In [ ]:
def sesion(tok, modelo, tarea, espacio, herramientas, mensajes, max_pasos, avisar=False,
           ver=False):
    """Una sesión del bucle del agente, con la puerta de las reglas del capítulo 2,
    sobre un repositorio que puede venir trabajado de antes. Con «avisar», dos pasos
    antes del final le pide que actualice el progreso. Devuelve la respuesta final
    (None si se acaban los pasos), los pasos y los tokens."""
    final, pasos, entrada, salida, rechazos = None, 0, 0, 0, 0
    for pasos in range(1, max_pasos + 1):
        if avisar and pasos == max_pasos - 1:
            mensajes.append({"role": "user", "content": AVISO_FIN})
        texto, recortados = recortar_conversacion(tok, mensajes)
        if recortados is None:
            break
        respuesta, n_entrada, n_salida = generar(tok, modelo, texto)
        entrada, salida = entrada + n_entrada, salida + n_salida
        llamadas = leer_llamadas(respuesta)
        if not llamadas:
            mensajes.append({"role": "assistant", "content": respuesta})
            if ver:
                print(f"[{pasos}] {respuesta}")
            if dice_terminado(respuesta) and rechazos < MAX_RECHAZOS:
                cambios = hr.diferencias(espacio.origen, espacio.carpeta)
                rotas = comprobar_reglas(tarea, herramientas.registro, cambios, respuesta)
                if rotas:
                    rechazos += 1
                    aviso = ("Todavía no está terminado según AGENTS.md: "
                             + "; ".join(f"{r}: {t}" for r, t in sorted(rotas.items()))
                             + ". Sigue trabajando.")
                    mensajes.append({"role": "user", "content": aviso})
                    continue
            final = respuesta
            break
        pensamiento = respuesta.split("<tool_call>")[0].strip()
        mensajes.append({"role": "assistant", "content": pensamiento,
                         "tool_calls": [{"type": "function", "function": l}
                                        for l in llamadas if l["name"]]})
        for llamada in llamadas:
            if llamada["name"] is None:
                resultado = {"error": "No se entiende la llamada a la herramienta."}
            else:
                resultado = herramientas.llamar(llamada["name"], llamada["arguments"])
            if ver:
                print(f"[{pasos}] {llamada['name']}"
                      f"({json.dumps(llamada['arguments'], ensure_ascii=False)[:90]})"
                      f" -> {json.dumps(resultado, ensure_ascii=False)[:110]}")
            mensajes.append({"role": "tool", "name": llamada["name"] or "?",
                             "content": json.dumps(resultado, ensure_ascii=False)})
    return final, pasos, entrada, salida


def agente(tok, modelo, tarea, sesiones=2, pasos_por_sesion=12, progreso=False,
           con_traspaso=False, ver=False):
    """Una tarea en una o dos sesiones sobre el mismo repositorio. Cada sesión empieza
    con la conversación vacía. Con «progreso», el AGENTS.md pide mantener
    .agente/PROGRESO.md; con «con_traspaso», la segunda sesión empieza con el informe
    de traspaso que prepara el programa. Devuelve la fila de resultados."""
    inicio = time.time()
    instrucciones = AGENTS_REGLAS + (REGLA_PROGRESO if progreso else "")
    sistema = SISTEMA + "\n\n" + instrucciones
    filas_sesion, final, ultimo = [], None, ""
    with hr.Espacio() as espacio:
        base = hr.Herramientas(espacio)
        for numero in range(1, sesiones + 1):
            peticion = tarea["peticion"]
            if numero > 1:
                peticion += ("\n\nOtra sesión ya empezó a trabajar en esta petición y se "
                             "cortó. Sigue desde donde lo dejó.")
                if progreso:
                    peticion += f" Lee {PROGRESO} antes de seguir."
                if con_traspaso:
                    peticion += "\n\n" + traspaso(espacio, ultimo)
            mensajes = [{"role": "system", "content": sistema},
                        {"role": "user", "content": peticion}]
            herramientas = HerramientasConReglas(base, tarea)  # sin lecturas previas
            antes = len(base.registro)
            if ver:
                print(f"--- sesión {numero}")
            final, pasos, entrada, salida = sesion(
                tok, modelo, tarea, espacio, herramientas, mensajes, pasos_por_sesion,
                avisar=progreso and numero < sesiones, ver=ver)
            ultimo = next((m["content"] for m in reversed(mensajes)
                           if m["role"] == "assistant" and m["content"]), "")
            filas_sesion.append({"pasos": pasos, "tokens_entrada": entrada,
                                 "tokens_salida": salida,
                                 "llamadas": base.registro[antes:]})
            if final is not None and dice_terminado(final):
                break                           # terminada: no hace falta otra sesión
        evaluacion = hr.evaluar_intento(espacio, tarea)
        cambios = hr.diferencias(espacio.origen, espacio.carpeta)
        rotas = comprobar_reglas(tarea, base.registro, cambios, final)
        tiene_progreso = espacio.ruta(PROGRESO).exists()
    terminado = final is not None and dice_terminado(final)
    segunda = filas_sesion[1]["llamadas"] if len(filas_sesion) > 1 else []
    leidos_antes = {r["argumentos"].get("ruta") for r in filas_sesion[0]["llamadas"]
                    if r["herramienta"] == "leer_archivo" and r["ok"]}
    return {"tarea": tarea["id"], "tipo": tarea["meta"]["tipo"],
            "resuelta": evaluacion["resuelta"], "dice_terminado": terminado,
            "terminado_falso": terminado and not evaluacion["resuelta"],
            "reglas_rotas": rotas, "sesiones": len(filas_sesion),
            "progreso_escrito": tiene_progreso,
            "relecturas": sum(r["herramienta"] == "leer_archivo"
                              and r["argumentos"].get("ruta") in leidos_antes
                              for r in segunda),
            "llamadas_segunda": len(segunda),
            "llamadas_hasta_editar": next(
                (i for i, r in enumerate(segunda, 1)
                 if r["herramienta"] in ("editar", "crear_archivo") and r["ok"]
                 and not r["argumentos"].get("ruta", "").startswith(".agente/")), None),
            "pasos": sum(s["pasos"] for s in filas_sesion),
            "tokens_entrada": sum(s["tokens_entrada"] for s in filas_sesion),
            "tokens_salida": sum(s["tokens_salida"] for s in filas_sesion),
            "segundos": round(time.time() - inicio, 1),
            "respuesta": final or "(sin respuesta final)",
            "llamadas": [[[r["herramienta"], r["argumentos"].get("ruta", ""), r["ok"]]
                          for r in s["llamadas"]] for s in filas_sesion],
            "diff": evaluacion["diff"]}

In [ ]:
for nombre in ("sin estado", "con traspaso"):
    print(f"=== {nombre}")
    fila = agente(tok, modelo, hr.leer_tarea("T17"), ver=True, **VARIANTES[nombre])
    print("Resuelta:", fila["resuelta"], "· Relecturas:", fila["relecturas"])

## 6. Las diecisiete tareas con las cuatro variantes

Tardan unas siete horas en una GPU de escritorio. Para una primera prueba, pasa `tareas=TAREAS_CAP3[:4]` a `probar`.

In [ ]:
TAREAS_CAP3 = TAREAS + ["T09", "T12", "T16"]   # con las tres de dos sesiones


VARIANTES = {
    "una sesión": {"sesiones": 1, "pasos_por_sesion": 24},
    "sin estado": {"sesiones": 2, "pasos_por_sesion": 12},
    "con progreso": {"sesiones": 2, "pasos_por_sesion": 12, "progreso": True},
    "con traspaso": {"sesiones": 2, "pasos_por_sesion": 12, "con_traspaso": True},
}


def resumen(filas):
    """Los totales de una variante."""
    dos = [f for f in filas if f["sesiones"] == 2]
    return {"tareas": len(filas),
            "resueltas": sum(f["resuelta"] for f in filas),
            "dice_terminado": sum(f["dice_terminado"] for f in filas),
            "terminado_falso": sum(f["terminado_falso"] for f in filas),
            "reglas_rotas": sum(len(f["reglas_rotas"]) for f in filas),
            "con_segunda_sesion": len(dos),
            "resueltas_en_segunda": sum(f["resuelta"] for f in dos),
            "relecturas": sum(f["relecturas"] for f in dos),
            "progreso_escrito": sum(f["progreso_escrito"] for f in filas),
            "pasos": sum(f["pasos"] for f in filas),
            "tokens_entrada": sum(f["tokens_entrada"] for f in filas),
            "segundos": round(sum(f["segundos"] for f in filas), 1)}


def probar(tok, modelo, opciones, tareas=TAREAS_CAP3):
    """Una variante con todas las tareas: sus filas y su resumen."""
    filas = []
    for identificador in tareas:
        fila = agente(tok, modelo, hr.leer_tarea(identificador), **opciones)
        filas.append(fila)
        print(f"  {identificador} {'resuelta' if fila['resuelta'] else 'no      '}"
              f"  {'dice terminado' if fila['dice_terminado'] else 'no lo dice    '}"
              f"  sesiones={fila['sesiones']}  relecturas={fila['relecturas']}"
              f"  pasos={fila['pasos']:2d}  {fila['segundos']:6.1f} s", flush=True)
    return {"filas": filas, "resumen": resumen(filas)}

In [ ]:
resultados = {}
for nombre, opciones in VARIANTES.items():
    print(f"== {nombre}")
    resultados[nombre] = probar(tok, modelo, opciones)
    print(resultados[nombre]["resumen"])

## Ejercicios

Los archivos de resultados que citan los ejercicios están en `datos/resultados/capitulo_3/`, que `preparar` descarga con los datos.

1. Lista lo que le haría falta saber a una persona que retoma una tarea a medias de un compañero. ¿Cuáles de esas cosas puede sacar un programa del repositorio y cuáles solo las sabe quien hizo el trabajo? *(Sin GPU.)*

2. Ejecuta la tarea del IVA (T04) con el traspaso y `ver=True`. Copia el informe que recibe la segunda sesión y señala qué línea le dice dónde está el error. *(GPU, unos 15 minutos.)*

3. Quita el diff del informe de traspaso y deja solo los archivos cambiados, las pruebas y el último mensaje. ¿Cuántas tareas resuelve? ¿Qué parte del informe pesa más? Los resultados con las diecisiete tareas ya están en `resultados_ejercicios.json`; para verlo tú, basta con tres, por ejemplo T04, T10 y T15. *(Sin GPU con los resultados guardados; GPU, unos 40 minutos, con esas tareas.)*

4. En la variante del progreso, el agente ignoró la regla R9 y el aviso. Escribe una comprobación que no deje acabar la primera sesión sin `.agente/PROGRESO.md`, como las reglas de proceso del capítulo 2. ¿Lo escribe entonces? ¿Ayuda a la segunda sesión? *(GPU, unos 40 minutos.)*

5. Divide las tareas en tres sesiones de ocho pasos con el traspaso. ¿Cuántas resuelve? ¿Dónde está el límite a partir del cual las sesiones son demasiado cortas? Los resultados con las diecisiete tareas ya están en `resultados_ejercicios.json`; para verlo tú, basta con tres, por ejemplo T04, T10 y T15. *(Sin GPU con los resultados guardados; GPU, unos 40 minutos, con esas tareas.)*

6. El informe recorta el diff a 2000 caracteres. Busca la tarea con el diff más largo al empezar la segunda sesión y mira qué se pierde con el recorte. Propón otra forma de resumir un diff largo. *(Sin GPU.)*

7. El traspaso incluye el último mensaje del agente. Busca un caso en el que ese mensaje diga algo falso sobre lo que hizo la primera sesión. ¿Se dio cuenta la segunda sesión al ver el diff? *(Sin GPU.)*

8. Las tres tareas de dos partes (pago a plazos, numeración de facturas y devoluciones de Navidad) no se resuelven con ninguna variante. Lee la traza de una de ellas con el traspaso y di si la segunda sesión llegó a la segunda parte. *(GPU, unos 20 minutos.)*

9. Calcula los tokens de entrada por tarea resuelta de las cuatro variantes. ¿Cuánto ocupa el informe de traspaso frente al resto de la conversación de la segunda sesión? *(Sin GPU.)*

10. **Proyecto.** Escribe un script de arranque para un repositorio tuyo que, al empezar a trabajar, muestre los cambios sin confirmar, el resultado de las pruebas y las últimas notas del historial de git. Úsalo una semana al empezar cada sesión con un agente de programación y anota si cambia cómo empieza.

## Referencias

Modelo [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). [Effective harnesses for long-running agents](https://www.anthropic.com/engineering/effective-harnesses-for-long-running-agents) (Anthropic, 2025). La tienda Ciclos Norte, su programa y sus tareas son ficticios. Código y experimentos propios.